<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# Module 5.5: The Official RISC-V Tests
**Prev: [A Single-Cycle RV32I Processor](5.4_single_cycle.ipynb)**<br>
**Next: [A Five-Stage Pipeline](5.6_pipeline.ipynb)**<br>

## Motivation
The core of [Module 5.4](5.4_single_cycle.ipynb) passes the tests we wrote, and retires the same
instructions as our golden model. But both were written by the same people, from the same reading
of the specification: if we misread an instruction, the model and the core agree, and are both
wrong. The [riscv-tests](https://github.com/riscv-software-src/riscv-tests) of RISC-V
International are an independent check: for each instruction, a program that runs it on chosen
operands (corner cases included) and compares the results with values computed elsewhere.

This notebook runs the `rv32ui` tests (the unprivileged RV32I instructions) on the golden model and
on the single-cycle core.

## Setup

In [ ]:
interp.load.module(os.pwd / "source" / "load-ivy.sc")

In [ ]:
interp.load.module(os.pwd / "source" / "load-riscv.sc")

In [ ]:
import chisel3._
import chisel3.simulator.PeekPokeAPI._
import riscv._

---
# How the tests got here

A riscv-test is an assembly file written with C preprocessor macros: `TEST_RR_OP(2, add, 3, 1, 2)`
expands to "load 1 and 2, add them, compare with 3, jump to `fail` if different", numbered so that a
failure says which case it was. The tests normally run in an *environment* (`riscv_test.h`) that
sets up trap handlers with CSRs, and they report through a memory location the simulator watches.

This course has neither CSRs nor a C compiler, so `tools/import-riscv-tests.py` (run on a
development machine, with git and `cpp`) did two things once: it replaced the environment with a
small CSR-free one, where a test starts at the first instruction and ends with `ecall`, leaving 0
in `a0` if it passed (or the failing case's number, shifted left by one, plus one); and it expanded
the macros with `cpp`. The expanded files are in `riscv/programs/riscv-tests/rv32ui/`, and `riscv.Asm`
assembles them here (they come out word for word as GNU `as` assembles them). Two tests are left
out: `fence_i` (`FENCE.I` belongs to the Zifencei extension) and `ma_data` (misaligned accesses,
which halt this course's cores).

The environment:

In [ ]:
val tests = os.pwd / "riscv" / "programs" / "riscv-tests"
println(os.read(tests / "riscv_test.h"))
println(os.read(tests / "EXCLUDED"))

And the first cases of `add` after expansion (`;` separates instructions on a line; `0xffff...`
values are written for 64-bit registers and masked to 32 bits by the expression around them):

In [ ]:
os.read.lines(tests / "rv32ui" / "add.S").filter(_.contains("test_")).take(3).foreach(println)

---
# On the golden model

`IsaSim` first: it needs no hardware, so this runs even before the exercises of 5.2-5.4 are done.
A test that the model fails would mean a misreading of the specification in the model itself.

In [ ]:
val all    = Programs.images("programs/riscv-tests/rv32ui") // each test, by name, assembled once
val names  = all.map(_._1)
val images = all.toMap
def outcome(halt: Option[Halt], a0: Long) =
  if (halt.contains(Halt.Ecall) && a0 == 0) "pass"
  else if (halt.contains(Halt.Ecall)) s"FAIL (case ${a0 >> 1})"
  else s"FAIL (stopped: ${halt.getOrElse("no halt")})"

val model = names.map { name =>
  val sim   = new IsaSim(images(name))
  val trace = sim.run(100000)
  name -> (outcome(sim.halted, sim.reg(riscv.Reg("a0"))), trace.size)
}.toMap
for (row <- names.grouped(4)) println(row.map(n => f"$n%-7s ${model(n)._1}%-5s").mkString("   "))
assert(model.values.forall(_._1 == "pass"))
println(s"${names.size} tests, ${model.values.map(_._2).sum} instructions: all pass on the golden model")

---
# On the single-cycle core

Now the core, in one simulation (the core is compiled once; `CoSim.load` writes each test into
the RAM and resets the core). For each test, `CoSim.compare` also checks every retired instruction
against the golden model, so a test passes here only if the core gets the right answer *the same
way*.

In [ ]:
val start = System.nanoTime
val core  = scala.collection.mutable.Map[String, String]()
simulate(new SingleCycleSystem) { c =>
  for (name <- names) {
    val difference = CoSim.compare(c, images(name))
    // no difference: the core halted as the golden model did, with ECALL, so a0 holds the result
    core(name) = difference.map(_ => "DIFFERS").getOrElse(outcome(Some(Halt.Ecall), CoSim.reg(c, riscv.Reg("a0"))))
    difference.foreach(d => println(s"$name: $d"))
  }
}
val seconds  = (System.nanoTime - start) / 1e9
val excluded = os.read.lines(tests / "EXCLUDED").map(_.takeWhile(_ != ':'))
println(f"${"test"}%-8s ${"golden model"}%-14s ${"core"}%-8s instructions")
for (name <- names) println(f"$name%-8s ${model(name)._1}%-14s ${core(name)}%-8s ${model(name)._2}")
for (name <- excluded) println(f"$name%-8s ${"unsupported"}%-14s ${"unsupported"}%-8s")
println(f"${names.count(core(_) == "pass")} of ${names.size} pass on the core, in $seconds%.0f s (simulation included)")
assert(names.forall(core(_) == "pass"))
println("SUCCESS!!")

Each core instruction takes one clock cycle, so the number of instructions is also the number of
cycles. Most of the time goes into compiling the simulation and into the testbench, which
peeks the retire port every cycle.

---
# You're done!

The single-cycle core passes the official tests. It is slow in another sense, though: its clock
period must cover the longest instruction, a load, from fetch to write back. The next notebooks
split that path into stages and run several instructions at once: a
[pipeline](5.6_pipeline.ipynb).

[Return to the top.](#top)